#### loss function 손실함수
: 모델의 예측값과 실제 정답이 얼마나 다른지 수치로 계산하는 함수(지도학습)
: 모델의 궁극적인 목표는 손실값을 줄이는 것

- 기본용어
    - Prediction: 모델이 예측한 값
    - Target: 정답
    - Loss: 예측값과 정답의 차이를 계산한 값

- 손실함수의 역할
    - 예측값과 정답의 차이를 수치로 표현
    - 모델이 얼마나 잘 예측했는지 평가할 수 있음
    - 자동미분을 통해weight와 bias를 업데이틀 할 때 사용할 기울기를 계산할 수 있음

#### 1. MSELoss(Mean Square Error Loss Function)
: 평균의 제곱 오차를 계산하는 손실함수
예측값과 정답의 차이를 제곱한 뒤 평균을 계산함

- 특징
    - 오차를 제곱하므로 오차가 클수록 더 큰 패널티(손실값)를 부여함
    - 손실값은 항상 0 이상임
    - 예측값이 정답과 같을 때 손실값은 0
    - 주로 회귀 문제에서 사용

- 사용방법
```python
loss_fn = nn.MSELoss()
loss_fn(prediction, target) 
# 순서가 바뀌어도 어차피 둘의 차이를 제곱하기에 상관없음
```

- 기본설정
    - reduction = 'mean'은 모든 원소의 제곱오차의 평균

In [1]:
import torch
import torch.nn as nn

In [4]:
pred = torch.tensor([2., 4, 6])
target = torch.tensor([1., 5, 4])

loss_fn = nn.MSELoss()
loss = loss_fn(pred, target)

print('예측값:', pred)
print('실제값:', target)
print('MSE Loss:', loss)

# 계산으로 검증
loss2 = ((target - pred) ** 2).mean()
print('직접 계산한 MSE Loss:', loss2)

예측값: tensor([2., 4., 6.])
실제값: tensor([1., 5., 4.])
MSE Loss: tensor(2.)
직접 계산한 MSE Loss: tensor(2.)


#### Reduction 옵션
| 옵션 | 의미 |
| --- | --- |
| mean | 모든 원소의 손실 평균(기본값) |
| sum | 모든 원소의 손실 합 |
| none | 원소별 손실을 그대로 반환 |

** none은 데이터별 손실을 따로 확인하거나 가중치를 적용할 때 유용함 **

In [45]:
pred = torch.tensor([2., 4, 6])
target = torch.tensor([1., 5, 4])

loss_mean = nn.MSELoss(reduction= 'mean')(pred, target) # default
loss_sum = nn.MSELoss(reduction= 'sum')(pred, target)
loss_none = nn.MSELoss(reduction= 'none')(pred, target)

print('오차평균:', loss_mean)
print('오차합:', loss_sum)
print('오차 그대로:', loss_none)

오차평균: tensor(2.)
오차합: tensor(6.)
오차 그대로: tensor([1., 1., 4.])


#### 2. L1Loss(Mean Absolute Error)
: 예측값과 정답의 차이를 절댓값을 적용한 뒤 평균 계산

- 특징
    - 오차에 절댓값 사용하여 손실값은 0 이상
    - 예측값과 정답이 같으면 손실값은 0
    - MSE 보다 큰 오차에 대한 패널티가 상대적으로 적음
    - Lasso Regression에 사용

- 사용법
```python
loss_fn = nn.L1loss()
loss = loss_fn(pred, target)
```

In [7]:
pred = torch.tensor([2., 4, 6])
target = torch. tensor([1., 5, 4])

loss_fn = nn.L1Loss()
loss = loss_fn(pred, target)

print('예측값:', pred)
print('실제값:', target)
print('L1Loss:', loss)  # MSE loss: tensor(2.) 보다 작음을 확인

# 계산으로 검증
loss2 = torch.abs(target - pred).mean()
print('직접 계산한 L1Loss:', loss2)

예측값: tensor([2., 4., 6.])
실제값: tensor([1., 5., 4.])
L1Loss: tensor(1.3333)
직접 계산한 L1Loss: tensor(1.3333)


- 큰 오차를 더 강하게 반영하고 싶다면 MSE
- 이상치의 영향을 상대적으로 적게 받고 싶으면 L1
- 오차가 많을 수록 패러티 값이 더 크기에 패러티 값을 줄이는 것을 목표로 하는 보통의 경우
    : MSE 더 많이 씀

In [9]:
prediction = torch.tensor([10., 10, 10])
target = torch.tensor([9., 8., 0])

mse_loss = nn.MSELoss()(prediction, target)
l1_loss = nn.L1Loss()(prediction, target)

print('MSE Loss:', mse_loss)
print('L1 Loss:', l1_loss)

# 오차가 큰 tensor[2]의 영향으로 MSE의 값이 훨씬 크다는 것을 확인

MSE Loss: tensor(35.)
L1 Loss: tensor(4.3333)


#### 3. BCEloss (Binary Cross Entropy)
: 이진분류에서 사용하는 손실 함수

**중요**
모델이 sigmoid를 통과한 확률값을 입력으로 받아야 함.

```ptyhon
nn.BCELoss()(probability, target)
```
probability 는 0과 1 사이의 확률값<br>
target은 정답. 일반적으로 0 또는 1이며, 반드시 실수형 텐서로 전달(정수 X)

- 실제 정답이 1인데 확률이 1에 가까우면 손실이 작아지고, 0에 가까워지면 손실이 커짐.
- 극단적인 값에서 Nan 발생 위험 있음

In [10]:
probability = torch.tensor([0.9, 0.2, 0.7, 0.1])
target = torch.tensor([1., 0, 1, 0])

loss_fn = nn.BCELoss()
loss = loss_fn(probability, target)

print('예측값:', probability)
print('실제값:', target)
print('BCE Loss 값:', loss)

# BCELoss = y * log(p) + (1-y) * log(1-p)
# y는 target 값
# p는 probability 값

예측값: tensor([0.9000, 0.2000, 0.7000, 0.1000])
실제값: tensor([1., 0., 1., 0.])
BCE Loss 값: tensor(0.1976)


In [12]:
import math
print(math.log(0.9))
print(torch.log(torch.tensor(0.9)))

-0.10536051565782628


#### 4. BCEWithLogitsLoss
: Sigmoid와 BCELoss 계산을 하나로 결합한 손실함수
- 극단적인 값에서 nan이 발생할 위험이 있는 BCE loss를 sigmoid로 보완

- 중요한 차이
    - BCEloss() : sigmoid를 적용한 확률값을 입력으로 받음
    - BCEWithLogitsLoss() : sigmoid를 적용하지 않은 Logits을 입력으로 받음

- 사용방식
```python
nn.BCEWithLogitsLoss()(logits, target)
```

- 사용하는 이유
: Sigmoid와 BCELoss계산을 결합하여 수치적으로 안정성이 높은 손실 계산이 가능
    - 극단적인 값에서 Nan 발생 위험이 적음

In [22]:
logits = torch.tensor([2., -1.5, 1, -2])
target = torch.tensor([1., 0, 1, 0])

prob = torch.sigmoid(logits) # all elements of input should be between 0 and 1
loss_fn1 = nn.BCELoss()
loss1 = loss_fn1(prob, target)
print('logits:', logits)
print('prob:', prob)
print('BCE loss after sigmoid activation:', loss1)
print()

loss_fn2 = nn.BCEWithLogitsLoss()
loss2 = loss_fn2(logits, target) # 바로 logit 입력
print('BCEWithLogitsloss:', loss2)

# 두 손실 값이 일치하는 지 확인 
# torch.allclose(비교값1, 비교값2)
print('Do they match?', torch.allclose(loss2, loss1))

logits: tensor([ 2.0000, -1.5000,  1.0000, -2.0000])
prob: tensor([0.8808, 0.1824, 0.7311, 0.1192])
BCE loss after sigmoid activation: tensor(0.1921)

BCEWithLogitsloss: tensor(0.1921)
Do they match? True


In [54]:
logits_a = torch.tensor([1000.], requires_grad= True)
target = torch.tensor([0.])

prob = torch.sigmoid(logits_a)  
# 1 / 1 + e^-1000  = 1보다 아주아주 작은 값으로 float32에 저장 불가 -> 1로 저장
loss_a = nn.BCELoss()(prob, target) # -log(1-1) = -log(0) = 무한대 -> 100으로 저장

logits_b = torch.tensor([1000.], requires_grad= True)
loss_b = nn.BCEWithLogitsLoss()(logits_b, target)
# L(x,y) = max(x,0) -x*y + log(1+e^(-1*|x|)) 라는 새로운 수식을 적용하여
# L = 1000 + 0 + 0 = 1000 이 저장됨

print(loss_a)
print(loss_b)

tensor(100., grad_fn=<BinaryCrossEntropyBackward0>)
tensor(1000., grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


In [31]:
# Loss 값이 차이나고, 기울기가 소실되는 예
target = torch.tensor([0.])

logits_a = torch.tensor([1000.], requires_grad= True)

prob = torch.sigmoid(logits_a)
loss_a = nn.BCELoss()(prob, target)

loss_a.backward()   # 기울기 소실되어 0 출력

logits_b = torch.tensor([1000.], requires_grad= True)

loss_b = nn.BCEWithLogitsLoss()(logits_b, target)

loss_b.backward()

print('Sigmoid 확률:', prob.item()) #.item(): tensor([###])을 ###로 출력 가능
print('BCELoss:', loss_a.item())
print('BCEWithLogitsLoss:', loss_b.item())

print('BCELoss Gradient:', logits_a.grad)   # 기울기 소실되어 0 출력
print('BCEWithLogitsLoss Gradient:', logits_b.grad) # 기울기 소실되지 않고 1 출력

Sigmoid 확률: 1.0
BCELoss: 100.0
BCEWithLogitsLoss: 1000.0
BCELoss Gradient: tensor([0.])
BCEWithLogitsLoss Gradient: tensor([1.])


#### 5. CrossEntropyLoss
: 여러 클래스 중 하나를 선택하는 다중 분류에서 주로 사용

- 사용방법
```python
nn.CrossEntropyLoss()(logits, target)
```
- 입력방법
    - 모델 출력값(logit): (batch_size, num_classes)
        - batch_size 넣어줄 배치의 크기
        - num_classes 분류될 클래스의 개수
    - 정답(target): (batch_size)
    - 정답의 자료형 : torch.long 

- 내부적으로 LogSoftmax와 NllLoss에 해당하는 계산을 수행
    - LogSoftmax : softmax 결과에 log를 적용한 함수
    - NlllLoss(Negative Log-Likelihood Loss) : 위 로그 확률을 이용하여 손실을 계산하는 함수
        - L = -log(p(y))

In [52]:
# 데이터 3개, 클래스 4개에 대한 모델 출력이 되었음(logits)
logits = torch.tensor([
    [2., 1, 0.1, 0.5],
    [0.2, 2.5, 1, 0.3],
    [0.1, 0.4, 0.8, 2]
])

# 각 데이터의 정답 클래스
target = torch.tensor([0, 1, 3], dtype=torch.long)

# loss 계산
loss = nn.CrossEntropyLoss()(logits, target)

# predict clas for each sample
pred = torch.argmax(logits, dim=1)

print('CrossEntropy Loss:', loss)
print('Predicted classes:', pred)

for i, predicted_class in enumerate(pred):
    print(f'Data {i + 1}: Class {predicted_class.item()}')

CrossEntropy Loss: tensor(0.4724)
Predicted classes: tensor([0, 1, 3])
Data 1: Class 0
Data 2: Class 1
Data 3: Class 3


#### 문제 유형에 따른 손실함수 선택
| 문제 유형 | 대표 손실함수 | 입력값 | 출력값 |
| --- | --- | --- | --- |
| 회귀 | MSELoss, L1Loss | 실수형 2차원 배열 | 실수형 2차원 텐서 |
| 이진분류 | BCELoss | Sigmoid 적용 확률 | 0 또는 1의 실수형 텐서|
| 이진분류 | BCEWithLogitsLoss | Sigmoid 적용 전 logits | 0 또는 1의 실수형 텐서|
| 다중분류 | CrossEntropyLoss | Sigmoid 적용 전 logits | 클래스 인덱스의 정수형 텐서(torch.long|

In [41]:
# 각 손실함수의 입력 형태에 따른 확인

# 회귀
regression_prediction = torch.tensor([2.5,3])
regression_target = torch.tensor([3, 2.])

print('MSE :', nn.MSELoss()(regression_prediction, regression_target).item())
print('L1 :', nn.L1Loss()(regression_prediction, regression_target).item())

# 이진분류
binary_logits = torch.tensor([1., -1])
binary_target = torch.tensor([1, 0.])

print('BCEWithLogits:', nn.BCEWithLogitsLoss()(binary_logits, binary_target).item())

# 다중 분류
multi_logits = torch.tensor([
    [1, 2, 0.5],
    [2, 0.5, 1]
])
multi_target = torch.tensor([1,0], dtype=torch.long)

print('CrossEntropy:', nn.CrossEntropyLoss()(multi_logits, multi_target).item())
print('CrossEntropy: 결과', torch.argmax(nn.CrossEntropyLoss()(multi_logits, multi_target)))

MSE : 0.625
L1 : 0.75
BCEWithLogits: 0.3132617175579071
CrossEntropy: 0.4643687903881073
CrossEntropy: 결과 tensor(0)
